In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [3]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 29 0 (offset 0)


Found 2 PDF files to process

Processing: First Password Creation.pdf
  ✓ Loaded 1 pages

Processing: 639233153835754249.pdf
  ✓ Loaded 1 pages

Total documents loaded: 2


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250721191230Z00'00'", 'moddate': "D:20250721191230Z00'00'", 'source': '../data/pdf/First Password Creation.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'First Password Creation.pdf', 'file_type': 'pdf'}, page_content='WELCOME TO NVIDIA!   \nCredential Format:  \nusername@nvidia.com  \n \nYour NVIDIA username/email address will be in \nthe email in which this document was \nattached.  \n \nDO NOT LOGIN UNTIL YOUR FIRST DAY  \n  \nTo set your first password:  \n1. On any web connected device browse to: https://aka.ms/sspr  \n2. Enter NVIDIA email and captcha code \n3. Select forgot password \n4. Select both methods for verification \na. Alternate Email Address and \nb. Text/Call Mobile   \n5. Enter verification codes sent via Phone & \nEmail  \n6. Create your first password   \n \n \n \n \n  \n  \n \n \n \n \n \nIf you need IT assistance, 

In [5]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [8]:
chunks=split_documents(all_pdf_documents)
chunks

Split 2 documents into 3 chunks

Example chunk:
Content: WELCOME TO NVIDIA!   
Credential Format:  
username@nvidia.com  
 
Your NVIDIA username/email address will be in 
the email in which this document was 
attached.  
 
DO NOT LOGIN UNTIL YOUR FIRST DAY ...
Metadata: {'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250721191230Z00'00'", 'moddate': "D:20250721191230Z00'00'", 'source': '../data/pdf/First Password Creation.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'First Password Creation.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 15.5 (Build 24F74) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20250721191230Z00'00'", 'moddate': "D:20250721191230Z00'00'", 'source': '../data/pdf/First Password Creation.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'First Password Creation.pdf', 'file_type': 'pdf'}, page_content='WELCOME TO NVIDIA!   \nCredential Format:  \nusername@nvidia.com  \n \nYour NVIDIA username/email address will be in \nthe email in which this document was \nattached.  \n \nDO NOT LOGIN UNTIL YOUR FIRST DAY  \n  \nTo set your first password:  \n1. On any web connected device browse to: https://aka.ms/sspr  \n2. Enter NVIDIA email and captcha code \n3. Select forgot password \n4. Select both methods for verification \na. Alternate Email Address and \nb. Text/Call Mobile   \n5. Enter verification codes sent via Phone & \nEmail  \n6. Create your first password   \n \n \n \n \n  \n  \n \n \n \n \n \nIf you need IT assistance, 

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

/Users/malleboyinavenkatakalyan/RAG-PipeLine/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18140.39it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/7h/3mbg80s13gx40f1_0kn9qqyr0000gn/T/ipykernel_30460/2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
